# Multi-Source ReAct Agentic RAG
## Web & Text Mining — Graduate Project Notebook
**Authors:** Mohammed Amine Rourrou & Khawla Chrifi Alaoui | **Program:** Big Data Analytics

---

## What this notebook builds

A complete, explainable **Multi-Source ReAct Agentic RAG** system that:
- Loads and indexes your **course lecture PDFs** and **research papers** using FAISS
- Exposes three retrieval tools to the agent: course search, research search, live web search
- Uses a **ReAct loop** (Reason → Act → Observe → Evaluate → Refine) to answer questions
- **Synthesizes answers from retrieved evidence** — not from hardcoded templates
- Compares itself against **naive RAG** to prove its superiority
- Produces **visualizations and analytics** on the retrieval pipeline

## Architecture
```
User Query
    ↓
[Planner]      detects which sources are needed
    ↓
[ReAct Loop]   Thought → Action (tool call) → Observation → Evaluation
    ├── search_course_materials()   FAISS on lecture PDFs
    ├── search_research_papers()    FAISS on academic papers  
    └── search_web()                Live Tavily search
    ↓
[Answer Synthesizer]   builds answer FROM retrieved text (not a template)
    ↓
Final Answer + Evidence Table + Source Citations
```

## Table of Contents
| # | Section |
|---|---------|
| 1 | Install dependencies |
| 2 | Imports and configuration |
| 3 | Validate folder structure |
| 4 | Load and extract documents |
| 5 | Fallback documents |
| 6 | Exploratory data analysis (EDA) |
| 7 | Text chunking |
| 8 | Chunk statistics and visualisation |
| 9 | Build FAISS vector stores |
| 10 | Test retrieval quality |
| 11 | Define retrieval tools |
| 12 | Evidence evaluation layer |
| 13 | ReAct planning and execution |
| 14 | Answer synthesizer |
| 15 | Full ReAct agent |
| 16 | Demo 1 — Course query |
| 17 | Demo 2 — Research query |
| 18 | Demo 3 — Multi-source query |
| 19 | Naive RAG vs Agentic RAG comparison |
| 20 | Results visualisation |
| 21 | Limitations and future work |
| 22 | Conclusion |


## Cell 1 — Install dependencies


In [ ]:
!pip install -q sentence-transformers faiss-cpu pypdf pandas numpy openai
!pip install -q tavily-python rich matplotlib seaborn
print("If you already installed, skip this cell.")

## Cell 2 — Imports and configuration

**Libraries used and why:**
- `sentence_transformers` — converts text to 384-dim semantic vectors
- `faiss` — Facebook's vector similarity search library (fast L2 nearest-neighbour)
- `pypdf` — extracts plain text from PDF files
- `pandas` — tabular display of retrieval results (good for Big Data Analytics)
- `anthropic` — Claude LLM for genuine answer synthesis from retrieved evidence
- `tavily` — reliable web search API designed for LLM agents
- `matplotlib / seaborn` — data visualisation

**Path detection:** the code automatically finds the project root whether the notebook
is inside a `Notebooks/` subfolder or at the project root level.

In [ ]:
import os, re, json, warnings, textwrap
from pathlib import Path
from typing import List, Dict, Any
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss

warnings.filterwarnings("ignore")

# ── Rich for coloured ReAct trace ─────────────────────────────────────────
try:
    from rich.console import Console
    from rich.panel import Panel
    from rich.table import Table
    from rich.rule import Rule
    console = Console()
    RICH = True
except ImportError:
    console = None
    RICH = False




import os

os.environ["OPENROUTER_API_KEY"] = ""
os.environ["TAVILY_API_KEY"] = ""
# ── API clients ────────────────────────────────────────────────────────────
import os

# 1) OpenRouter LLM client
try:
    from openai import OpenAI

    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

    if OPENROUTER_API_KEY:
        LLM_CLIENT = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=OPENROUTER_API_KEY
        )
        LLM_AVAILABLE = True
        print("✅ OpenRouter LLM client initialized.")
    else:
        LLM_CLIENT = None
        LLM_AVAILABLE = False
        print("⚠️ OPENROUTER_API_KEY not found. LLM synthesis will use fallback mode.")

except ImportError:
    LLM_CLIENT = None
    LLM_AVAILABLE = False
    print("⚠️ openai package not installed. Run: pip install openai")


# 2) Tavily web search client
try:
    from tavily import TavilyClient

    TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY")

    if TAVILY_API_KEY:
        WEB_CLIENT = TavilyClient(api_key=TAVILY_API_KEY)
        WEB_AVAILABLE = True
        print("✅ Tavily web search client initialized.")
    else:
        WEB_CLIENT = None
        WEB_AVAILABLE = False
        print("⚠️ TAVILY_API_KEY not found. Web search will be disabled.")

except ImportError:
    WEB_CLIENT = None
    WEB_AVAILABLE = False
    print("⚠️ tavily-python package not installed. Run: pip install tavily-python")

# ── Project paths ──────────────────────────────────────────────────────────
CURRENT_DIR  = Path.cwd()
PROJECT_ROOT = (CURRENT_DIR.parent
                if CURRENT_DIR.name.lower() in {"notebooks", "notebook"}
                else CURRENT_DIR)
COURSE_DIR   = PROJECT_ROOT / "data" / "course_materials"
RESEARCH_DIR = PROJECT_ROOT / "data" / "research_papers"

# ── Matplotlib style ───────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 11
})
sns.set_palette("husl")

print(f"✓ Project root : {PROJECT_ROOT.name}/")
print(f"✓ Course folder: {COURSE_DIR.name}/")
print(f"✓ Research dir : {RESEARCH_DIR.name}/")
print(f"✓ LLM ready    : {LLM_AVAILABLE}")
print(f"✓ Web search   : {WEB_AVAILABLE}")

## Cell 3 — Validate folder structure
Confirms folders exist and lists detected files before loading anything.

In [ ]:
def validate_folders():
    for folder, label in [(COURSE_DIR, "course_materials"),
                          (RESEARCH_DIR, "research_papers")]:
        exists = folder.exists()
        files  = list(folder.glob("*")) if exists else []
        pdfs   = [f for f in files if f.suffix.lower() in {".pdf",".txt",".md"}]
        status = "✅" if pdfs else ("⚠️  empty" if exists else "❌ missing")
        print(f"{status}  {label}/  ({len(pdfs)} usable files)")
        for f in pdfs[:8]:
            print(f"    • {f.name}")

validate_folders()

## Cell 4 — Load and extract documents

**Text extraction pipeline:**
1. `PdfReader` reads each page and joins extracted text
2. Whitespace is normalised (`\s+` → single space)
3. Files with fewer than 100 characters are skipped (likely scanned images)
4. Each document is stored as a dict: `{source_type, filename, text}`

**Supported formats:** `.pdf`, `.txt`, `.md`

In [ ]:
def extract_pdf(path: Path) -> str:
    text = ""
    try:
        for page in PdfReader(str(path)).pages:
            t = page.extract_text()
            if t: text += t + "\n"
    except Exception as e:
        print(f"  ⚠️  {path.name}: {e}")
    return text

def load_folder(folder: Path, source_type: str) -> List[Dict]:
    docs, supported = [], {".pdf", ".txt", ".md"}
    if not folder.exists():
        return docs
    for f in sorted(folder.iterdir()):
        if f.suffix.lower() not in supported:
            continue
        raw  = extract_pdf(f) if f.suffix.lower() == ".pdf" else f.read_text(errors="ignore")
        text = re.sub(r"\s+", " ", raw).strip()
        if len(text) < 100:
            print(f"  ⚠️  Very little text from {f.name} — skipping")
            continue
        docs.append({"source_type": source_type, "filename": f.name, "text": text})
        print(f"  ✓  {f.name}  ({len(text):,} chars)")
    return docs

print("Loading course materials...")
course_docs = load_folder(COURSE_DIR, "course_material")
print(f"  → {len(course_docs)} document(s)\n")

print("Loading research papers...")
research_docs = load_folder(RESEARCH_DIR, "research_paper")
print(f"  → {len(research_docs)} document(s)")

## Cell 5 — Fallback documents

This step provides small fallback text samples in case no documents are detected in the folders.  
In our experiments, the system mainly uses the lecture materials and selected research papers added to the dataset folders.

In [ ]:
FALLBACK_COURSE = [
    {"source_type":"course_material","filename":"ir_models.txt","text":
     "Information Retrieval is a core topic in Web and Text Mining. It covers Boolean retrieval, "
     "TF-IDF, BM25, vector space models, and semantic search with embeddings. "
     "TF-IDF weights terms by frequency in a document (TF) and rarity across the corpus (IDF). "
     "BM25 extends TF-IDF with document length normalisation. "
     "Vector space models represent documents as high-dimensional vectors; "
     "cosine similarity measures relevance between query and document vectors."},
    {"source_type":"course_material","filename":"pagerank.txt","text":
     "PageRank ranks web pages by analysing the hyperlink graph. "
     "A page is important if linked to by other important pages. "
     "PR(A) = (1-d) + d * sum(PR(Ti)/C(Ti)). The damping factor d is typically 0.85. "
     "PageRank is computed iteratively until convergence. "
     "It underpins web structure mining and was the original Google ranking algorithm."},
    {"source_type":"course_material","filename":"rag_intro.txt","text":
     "Retrieval-Augmented Generation (RAG) combines vector retrieval with LLM generation. "
     "The pipeline: embed query → retrieve top-k chunks → build prompt with context → generate answer. "
     "Limitations of naive RAG: single source, one-shot retrieval, no reasoning, no web access."},
    {"source_type":"course_material","filename":"text_mining.txt","text":
     "Text mining extracts patterns from unstructured text. "
     "Core steps: tokenisation, stop-word removal, stemming/lemmatisation, feature extraction. "
     "Topic modelling (LDA) discovers latent topics. "
     "Named Entity Recognition (NER) extracts people, places, organisations. "
     "Sentiment analysis classifies polarity. Document clustering groups similar texts."},
]

FALLBACK_RESEARCH = [
    {"source_type":"research_paper","filename":"ReAct_Yao2022.txt","text":
     "ReAct: Synergizing Reasoning and Acting in Language Models (Yao et al., ICLR 2023). "
     "ReAct interleaves reasoning traces (Thought) with tool calls (Action) and results (Observation). "
     "The loop: Thought → Action → Observation → Thought → ... → Final Answer. "
     "ReAct outperforms chain-of-thought and acts-only baselines on HotpotQA and FEVER."},
    {"source_type":"research_paper","filename":"RAG_Lewis2020.txt","text":
     "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (Lewis et al., NeurIPS 2020). "
     "RAG combines a pre-trained seq2seq model with dense passage retrieval over Wikipedia. "
     "The retriever encodes queries into dense vectors and finds top-k passages via MIPS. "
     "The generator conditions on the query and retrieved passages to produce the final answer."},
    {"source_type":"research_paper","filename":"SELF_RAG_Asai2023.txt","text":
     "Self-RAG (Asai et al., 2023) fine-tunes the LLM to generate reflection tokens inline. "
     "Tokens: [Retrieve] — should I retrieve now? [IsRel] — is this passage relevant? "
     "[IsSup] — does the passage support my generation? [IsUse] — is my answer useful? "
     "Unlike external orchestration, the model itself controls when and what to retrieve."},
    {"source_type":"research_paper","filename":"AgenticRAG_survey.txt","text":
     "Agentic RAG extends RAG with planning, tool selection, query decomposition, "
     "iterative retrieval, self-reflection, and evidence evaluation. "
     "Instead of retrieving once, the agent decides which source to query and "
     "whether retrieved evidence is sufficient, looping back if necessary. "
     "Frameworks include LangGraph, LlamaIndex Agents, and AutoGPT-style systems."},
    {"source_type":"research_paper","filename":"AgenticRAG_limits.txt","text":
     "Limitations of Agentic RAG: latency increases with each retrieval iteration; "
     "compounding retrieval errors reduce precision across hops; "
     "the agent may hallucinate tool calls or reasoning steps; "
     "context window limits constrain how much retrieved text can be used; "
     "the stopping criterion (when to stop retrieving) remains an open research problem."},
]

if not course_docs:
    print("⚠️  No course PDFs found — using built-in fallback text")
    course_docs = FALLBACK_COURSE
if not research_docs:
    print("⚠️  No research PDFs found — using built-in fallback text")
    research_docs = FALLBACK_RESEARCH

print(f"✓  {len(course_docs)} course doc(s)  |  {len(research_docs)} research doc(s)")

## Cell 6 — Exploratory Data Analysis (EDA)

Before building the pipeline, we need to understand the data:
- How many documents per source type?
- How long are they (character and word counts)?
- What is the vocabulary distribution?


In [ ]:
all_docs = course_docs + research_docs
df_docs  = pd.DataFrame([{
    "source_type": d["source_type"],
    "filename":    d["filename"],
    "chars":       len(d["text"]),
    "words":       len(d["text"].split()),
} for d in all_docs])

print("=== Document corpus overview ===")
print(df_docs.groupby("source_type")[["chars","words"]].agg(["count","mean","max"]).round(0))
print()
print("Full table:")
display(df_docs[["source_type","filename","words"]].to_string(index=False))

# ── Plot: document word counts by source type ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = {"course_material": "#6366F1", "research_paper": "#10B981"}
for source, grp in df_docs.groupby("source_type"):
    axes[0].barh(grp["filename"].str[:35], grp["words"],
                 color=colors.get(source, "#888"), label=source, alpha=0.85)
axes[0].set_xlabel("Word count")
axes[0].set_title("Document sizes")
axes[0].legend(fontsize=9)

counts = df_docs["source_type"].value_counts()
axes[1].pie(counts, labels=counts.index, autopct="%1.0f%%",
            colors=[colors.get(s,"#888") for s in counts.index],
            startangle=90, textprops={"fontsize":11})
axes[1].set_title("Documents by source type")

plt.tight_layout()
plt.savefig("eda_documents.png", bbox_inches="tight")
plt.show()
print("✓ Plot saved as eda_documents.png")

## Cell 7 — Text chunking

**Why chunk?**
Embedding an entire 50-page PDF as one vector averages all its content into a single point.
Searching it returns "this document is broadly related" — not "this paragraph answers the question."

**Chunking strategy:**
- `chunk_size = 300 words` — roughly one dense paragraph, fits well in a prompt
- `overlap = 50 words` — adjacent chunks share 50 words so no sentence is cut mid-thought
- `step = 250 words` — how far the window advances each iteration

A 5,000-word document produces approximately 20 chunks.

In [ ]:
def chunk_text(text: str, chunk_size=300, overlap=50) -> List[str]:
    words  = text.split()
    step   = chunk_size - overlap
    chunks = []
    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + chunk_size])
        if len(chunk) > 60:        # skip tiny trailing fragments
            chunks.append(chunk)
    return chunks

def build_chunks(docs: List[Dict]) -> List[Dict]:
    all_chunks = []
    for doc in docs:
        for idx, chunk_text_ in enumerate(chunk_text(doc["text"])):
            all_chunks.append({
                "source_type": doc["source_type"],
                "filename":    doc["filename"],
                "chunk_id":    idx,
                "text":        chunk_text_,
            })
    return all_chunks

course_chunks   = build_chunks(course_docs)
research_chunks = build_chunks(research_docs)
all_chunks      = course_chunks + research_chunks

print(f"✓ Course chunks   : {len(course_chunks)}")
print(f"✓ Research chunks : {len(research_chunks)}")
print(f"✓ Total chunks    : {len(all_chunks)}")
print(f"\nSample chunk (course, idx 0):")
print(textwrap.fill(course_chunks[0]["text"][:300], width=90))

## Cell 8 — Chunk statistics and visualization

Before generating embeddings, we analyze the chunked corpus to better understand the distribution of chunk sizes across the dataset.

This analysis is important because:
- Very short chunks may not contain sufficient semantic context for reliable retrieval
- Very large chunks may introduce irrelevant information and reduce embedding quality

In [ ]:
df_chunks = pd.DataFrame([{
    "source_type": c["source_type"],
    "filename":    c["filename"],
    "chunk_id":    c["chunk_id"],
    "words":       len(c["text"].split()),
    "chars":       len(c["text"]),
} for c in all_chunks])

print("=== Chunk corpus statistics ===")
print(df_chunks.groupby("source_type")[["words","chars"]].describe().round(1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Histogram: word count distribution ────────────────────────────────────
for source, grp in df_chunks.groupby("source_type"):
    axes[0].hist(grp["words"], bins=25, alpha=0.7,
                 label=source, color=colors.get(source,"#888"))
axes[0].set_xlabel("Words per chunk")
axes[0].set_ylabel("Count")
axes[0].set_title("Chunk size distribution")
axes[0].legend(fontsize=9)

# ── Bar: chunks per document (top 12) ─────────────────────────────────────
top = df_chunks.groupby("filename").size().sort_values(ascending=False).head(12)
bar_colors = [colors.get(
    df_chunks[df_chunks["filename"]==f]["source_type"].iloc[0], "#888")
    for f in top.index]
axes[1].barh(top.index.str[:30], top.values, color=bar_colors, alpha=0.85)
axes[1].set_xlabel("Number of chunks")
axes[1].set_title("Chunks per document")
patches = [mpatches.Patch(color=v, label=k) for k,v in colors.items()]
axes[1].legend(handles=patches, fontsize=9)

# ── Stacked bar: source type breakdown ────────────────────────────────────
counts = df_chunks.groupby("source_type").size()
axes[2].bar(counts.index, counts.values,
            color=[colors.get(s,"#888") for s in counts.index], alpha=0.85)
axes[2].set_ylabel("Total chunks")
axes[2].set_title("Chunks by source type")
for i, (idx, val) in enumerate(counts.items()):
    axes[2].text(i, val + 1, str(val), ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("eda_chunks.png", bbox_inches="tight")
plt.show()
print("✓ Plot saved as eda_chunks.png")

## Cell 9 — Build FAISS vector stores

**Embedding model:** `all-MiniLM-L6-v2`
- 384 dimensions, fast, runs on CPU, ~80 MB download (cached after first run)
- Good quality for academic/technical text

**FAISS IndexFlatL2:**
- Stores all 384-dim float32 vectors
- `search(query_vec, k)` returns the k vectors with smallest L2 distance
- For normalised vectors: `cosine_similarity = 1 - (L2_distance² / 2)`
- Exact search — no approximation, correct for corpora under ~100k chunks

**Two separate indexes** — one for course, one for research — so the agent can
choose which source to query rather than blending everything together.

In [ ]:
class FAISSStore:
    """
    Vector store backed by FAISS IndexFlatL2.
    Embeddings are L2-normalised so that cosine similarity
    can be computed correctly from L2 distance.
    """
    def __init__(self):
        self.model      = None   # loaded lazily (shared across instances)
        self.index      = None
        self.chunks     : List[Dict] = []
        self.embeddings : np.ndarray = None

    def build(self, chunks: List[Dict], model: SentenceTransformer):
        self.model  = model
        self.chunks = chunks
        texts = [c["text"] for c in chunks]

        print(f"  Embedding {len(texts)} chunks...")
        embs = model.encode(texts, convert_to_numpy=True,
                             show_progress_bar=True, normalize_embeddings=True,
                             batch_size=32).astype("float32")
        self.embeddings = embs
        dim = embs.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(embs)
        print(f"  ✓ FAISS index: {self.index.ntotal} vectors, dim={dim}")

    def search(self, query: str, k: int = 5) -> List[Dict]:
        """Return top-k chunks with correct cosine similarity scores."""
        q_emb = self.model.encode([query], convert_to_numpy=True,
                                   normalize_embeddings=True).astype("float32")
        distances, indices = self.index.search(q_emb, min(k, self.index.ntotal))
        results = []
        for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
            if idx < 0:
                continue
            item = self.chunks[idx].copy()
            item["rank"]       = rank
            item["l2_dist"]    = float(dist)
            # Correct cosine similarity for normalised L2 vectors
            item["similarity"] = float(1.0 - dist / 2.0)
            results.append(item)
        return results

    @property
    def size(self): return self.index.ntotal if self.index else 0


# ── Load model once, share across both stores ─────────────────────────────
print("Loading sentence-transformer model (cached after first run)...")
EMBEDDER = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✓ Model ready — embedding dim: {EMBEDDER.get_sentence_embedding_dimension()}")

print("\nBuilding course vector store...")
course_store = FAISSStore()
course_store.build(course_chunks, EMBEDDER)

print("\nBuilding research vector store...")
research_store = FAISSStore()
research_store.build(research_chunks, EMBEDDER)

print(f"\n✓ Total vectors indexed: {course_store.size + research_store.size}")

## Cell 10 — Retrieval quality evaluation

Before integrating the retrieval module into the full agentic pipeline, we evaluate the quality of the vector stores by testing whether relevant chunks are correctly retrieved for sample queries.

This step helps assess the effectiveness of the semantic retrieval process.

For relevant queries, higher cosine similarity scores generally indicate better semantic alignment between the query and the retrieved chunks.

In [ ]:
def show_retrieval(query: str, store: FAISSStore, label: str, k: int = 4):
    print(f"\n{'─'*70}")
    print(f"Query: {query}")
    print(f"Store: {label}  ({store.size} vectors)")
    print(f"{'─'*70}")
    results = store.search(query, k=k)
    rows = []
    for r in results:
        rows.append({
            "Rank":       r["rank"],
            "Source":     r["filename"][:35],
            "Chunk":      r["chunk_id"],
            "Similarity": f"{r['similarity']:.3f}",
            "Preview":    r["text"][:80] + "…",
        })
    df = pd.DataFrame(rows)
    display(df)
    return results

# ── Test queries ──────────────────────────────────────────────────────────
show_retrieval("What is TF-IDF and how is it used in information retrieval?",
               course_store, "Course Materials")
show_retrieval("What are the limitations of Agentic RAG?",
               research_store, "Research Papers")

## Cell 11 — Retrieval tools

Three functions the ReAct agent can call.
Each returns a uniform list of result dicts so the agent processes them identically.

| Tool | Source | When to use |
|---|---|---|
| `search_course_materials` | FAISS course index | Concepts from lectures, course-specific definitions |
| `search_research_papers` | FAISS research index | Academic findings, paper citations, method details |
| `search_web` | Tavily live search | Recent news, real-world applications, current data |

**Tavily** is used instead of DuckDuckGo because it is designed for LLM agents,
returns structured clean content (not raw HTML), and is reliable for live demos.

In [ ]:
def search_course_materials(query: str, k: int = 5) -> List[Dict]:
    """Search the FAISS index of course lecture materials."""
    return course_store.search(query, k=k)


def search_research_papers(query: str, k: int = 5) -> List[Dict]:
    """Search the FAISS index of academic research papers."""
    return research_store.search(query, k=k)


def search_web(query: str, k: int = 4) -> List[Dict]:
    """Live web search via Tavily. Falls back gracefully if unavailable."""
    if not WEB_AVAILABLE:
        return [{"source_type":"web","rank":1,"filename":"web_unavailable",
                 "chunk_id":0,"text":"Web search not configured. Add TAVILY_API_KEY.",
                 "similarity":None}]
    try:
        hits = WEB_CLIENT.search(
            query=query,
            search_depth="basic",
            max_results=k
        )["results"]

        return [{
            "source_type": "web",
            "rank":        i + 1,
            "filename":    h.get("title","Web result"),
            "chunk_id":    i,
            "url":         h.get("url",""),
            "text":        h.get("content","")[:600],
            "similarity":  None,
        } for i, h in enumerate(hits) if h.get("content","")]

    except Exception as e:
        return [{"source_type":"web","rank":1,"filename":"web_error",
                 "chunk_id":0,"text":f"Web search failed: {e}","similarity":None}]


TOOL_REGISTRY = {
    "course":   search_course_materials,
    "research": search_research_papers,
    "web":      search_web,
}
print("✓ Tools registered:", list(TOOL_REGISTRY.keys()))

## Cell 12 — Evidence evaluation layer

This layer is what makes the system **stronger than naive RAG**.

After each retrieval pass, it checks:
1. Did we retrieve from every source the query requires?
2. Is the best similarity score above the quality threshold?
3. If not — which tools should we call next?

The agent loops back to retrieve more only when this layer says evidence is insufficient.
This prevents both under-retrieval (incomplete answers) and over-retrieval (wasted calls).

In [ ]:
MIN_SIMILARITY = 0.35   # threshold below which local evidence is considered weak

def detect_needs(query: str) -> Dict[str, bool]:
    """
    Detect which sources the query likely needs.
    Uses keyword signals — honest about being rule-based.
    """
    q = query.lower()
    course   = any(w in q for w in ["course","lecture","class","slides","from our","our course"])
    research = any(w in q for w in ["research","paper","literature","academic","survey","study","according to"])
    web      = any(w in q for w in ["recent","latest","current","real-world","application","2024","2025","2026","industry"])
    # Default: search both local sources if no explicit signal
    if not (course or research or web):
        course = research = True
    return {"course": course, "research": research, "web": web}


def evaluate_evidence(query: str, evidence: List[Dict]) -> Dict:
    """
    Assess whether retrieved evidence is sufficient to answer the query.
    Returns: {sufficient, needs, found_types, missing, best_similarity, reasons}
    """
    needs       = detect_needs(query)
    found_types = {e.get("source_type") for e in evidence}
    local_sims  = [e.get("similarity") or 0 for e in evidence
                   if e.get("source_type") in {"course_material","research_paper"}]
    best_sim    = max(local_sims, default=0)

    missing = []
    if needs["course"]   and "course_material" not in found_types: missing.append("course")
    if needs["research"] and "research_paper"  not in found_types: missing.append("research")
    if needs["web"]      and "web"             not in found_types: missing.append("web")

    weak    = bool(local_sims) and best_sim < MIN_SIMILARITY
    sufficient = not missing and not weak

    reasons = (["Evidence is sufficient."] if sufficient else
               [f"Missing: {', '.join(missing)}" if missing else "",
                f"Low similarity: {best_sim:.3f}" if weak else ""])
    reasons = [r for r in reasons if r]

    return {"sufficient": sufficient, "needs": needs,
            "found_types": list(found_types), "missing": missing,
            "best_similarity": best_sim, "reasons": reasons}


print("✓ Evidence evaluator ready")
print(f"  Min similarity threshold : {MIN_SIMILARITY}")

## Cell 13 — ReAct planning and execution

**The ReAct loop:**
```
Thought  → understand query, decide which tools are needed
Action   → call a tool and retrieve evidence
Observe  → read what came back
Evaluate → is this sufficient? any missing sources?
Refine   → if not sufficient, loop back to Action
Answer   → synthesize from evidence
```

This cell defines the planning logic and the coloured trace printer.
The trace makes every decision visible — which is the best part of the demo.

In [ ]:
def plan_tools(query: str) -> List[str]:
    needs = detect_needs(query)
    return [t for t in ["course","research","web"] if needs.get(t)]


def extra_tools(evaluation: Dict, used: List[str]) -> List[str]:
    return [t for t in evaluation["missing"] if t not in used]


# ── Rich trace printer ────────────────────────────────────────────────────
STEP_STYLES = {
    "Thought":     ("blue",   "💭"),
    "Action":      ("yellow", "⚡"),
    "Observation": ("green",  "👁 "),
    "Evaluation":  ("magenta","⚖️ "),
    "Answer":      ("cyan",   "✅"),
}

def print_step(step_type: str, message: str):
    if RICH:
        icon  = STEP_STYLES[step_type][1]
        color = STEP_STYLES[step_type][0]
        console.print(Panel(f"{icon}  {message}",
                            title=f"[{color} bold]{step_type}[/{color} bold]",
                            border_style=color, expand=False))
    else:
        print(f"[{step_type}] {message}")


def evidence_table(evidence: List[Dict]):
    """Display retrieved evidence as a pandas DataFrame."""
    rows = []
    for e in evidence:
        rows.append({
            "source_type": e.get("source_type","?"),
            "filename":    (e.get("filename","?"))[:38],
            "chunk_id":    e.get("chunk_id","?"),
            "similarity":  f"{e['similarity']:.3f}" if e.get("similarity") else "—",
            "preview":     e.get("text","")[:70] + "…",
        })
    if rows:
        display(pd.DataFrame(rows))


print("✓ Planner and trace printer ready")

## Cell 14 — Answer synthesizer

This cell contains **two synthesis modes:**

**Mode A — LLM synthesis (if Anthropic key is set):**
Sends the query + retrieved evidence to Claude. Claude reads the actual retrieved text
and writes a genuine, query-specific answer grounded in that content.
This is true generative AI — every answer is different.

**Mode B — Extractive synthesis (no API key needed):**
Formats the top retrieved chunks directly as the answer, clearly attributing each passage.
Honest and transparent — shows the retrieved content without hallucinating.

Both modes cite sources. Mode A produces more fluent prose; Mode B is more transparent.

In [ ]:

LLM_SYSTEM = """
You are a research assistant.

Answer the question using ONLY the provided evidence.

Rules:
- Use retrieved evidence only
- Cite filenames or web titles
- Be specific and structured
- If evidence is insufficient, explicitly say so
""".strip()


def synthesize_with_llm(query: str, evidence: List[Dict]) -> str:
    """Generate grounded answer using OpenRouter."""

    context = ""

    for i, e in enumerate(evidence[:8], 1):

        source_name = e.get("filename", "source")
        source_type = e.get("source_type", "unknown")
        url = e.get("url", "")

        context += f"""
[Source {i}]
Type: {source_type}
Name: {source_name}
URL: {url}

Content:
{e.get('text', '')[:1000]}

"""

    prompt = f"""
Question:
{query}

Evidence:
{context}

Generate a grounded answer using ONLY the evidence above.
"""

    try:

        response = LLM_CLIENT.chat.completions.create(

            model="deepseek/deepseek-chat",

            messages=[
                {
                    "role": "system",
                    "content": LLM_SYSTEM
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=0.2,
            max_tokens=900
        )

        return response.choices[0].message.content.strip()

    except Exception as e:

        return f"LLM synthesis failed: {e}"



In [ ]:
def generate_answer(query: str, evidence: List[Dict], evaluation: Dict) -> str:
    """
    Produce the final answer using either:
    1. OpenRouter LLM synthesis, if an API key is available
    2. Extractive fallback, if no LLM key is available
    """

    if LLM_AVAILABLE:
        body = synthesize_with_llm(query, evidence)
        mode = "LLM via OpenRouter"
    else:
        body = synthesize_extractive(query, evidence)
        mode = "Extractive fallback"

    confidence = "High" if evaluation.get("sufficient", False) else "Medium — evidence may be incomplete"

    source_lines = []
    seen = set()

    for e in evidence:
        key = e.get("url") or (e.get("filename"), e.get("chunk_id"))

        if key in seen:
            continue

        seen.add(key)

        source_type = e.get("source_type", "unknown")

        if source_type == "web":
            source_lines.append(
                f"  • [Web] {e.get('filename', e.get('title', 'Web result'))} — {e.get('url', '')}"
            )
        else:
            source_lines.append(
                f"  • [{source_type}] {e.get('filename', '?')} — chunk {e.get('chunk_id', '?')}"
            )

    header = f"# Final Answer ({confidence} confidence | {mode})\n\n"

    footer = "\n\n## Sources Used\n"
    footer += "\n".join(source_lines[:10]) if source_lines else "No sources were retrieved."

    return header + body.strip() + footer


print("✓ Answer synthesizer ready")
print(f"  Mode: {'LLM via OpenRouter' if LLM_AVAILABLE else 'Extractive fallback'}")

## Cell 15 — Full ReAct agent

This is the orchestrator that connects all previous components into a single callable function.

**`run_react_agent(query)` does:**
1. Plan: detect which tools are needed
2. Loop: call each tool → observe → evaluate
3. Refine: call extra tools if evidence is insufficient
4. Answer: synthesize from collected evidence

**Returns:** `{answer, evidence, trace, tools_used, steps, evaluation}`

In [ ]:
def run_react_agent(query: str, max_extra_steps: int = 2, verbose: bool = True) -> Dict:
    trace, evidence, tools_used = [], [], []

    def log(step_type, msg):
        trace.append({"step": step_type, "message": msg})
        if verbose:
            print_step(step_type, msg)

    # ── Step 1: Plan ──────────────────────────────────────────────────────
    planned = plan_tools(query)
    log("Thought", f"Query analysis complete. Planned tools: {planned}")

    # ── Step 2: Execute planned tools ─────────────────────────────────────
    for tool_name in planned:
        log("Action", f"Calling tool: search_{tool_name}( query )")
        results = TOOL_REGISTRY[tool_name](query)
        evidence.extend(results)
        tools_used.append(tool_name)
        log("Observation", f"Retrieved {len(results)} result(s) from '{tool_name}'.")

    # ── Step 3: Evaluate evidence ──────────────────────────────────────────
    evaluation = evaluate_evidence(query, evidence)
    log("Evaluation", " | ".join(evaluation["reasons"]))

    # ── Step 4: Refine if needed ──────────────────────────────────────────
    extra = extra_tools(evaluation, tools_used)
    for step, tool_name in enumerate(extra[:max_extra_steps]):
        log("Thought", f"Evidence incomplete. Searching additional source: '{tool_name}'.")
        log("Action", f"Calling tool: search_{tool_name}( query )")
        results = TOOL_REGISTRY[tool_name](query)
        evidence.extend(results)
        tools_used.append(tool_name)
        log("Observation", f"Retrieved {len(results)} additional result(s).")
        evaluation = evaluate_evidence(query, evidence)
        log("Evaluation", " | ".join(evaluation["reasons"]))
        if evaluation["sufficient"]:
            break

    # ── Step 5: Synthesize answer ─────────────────────────────────────────
    log("Thought", "Evidence collected. Generating final answer.")
    answer = generate_answer(query, evidence, evaluation)

    if verbose:
        print_step("Answer", f"Answer generated — {len(evidence)} chunks from {len(set(tools_used))} source type(s).")
        print("\n─── Evidence table ───")
        evidence_table(evidence)
        print()
        from IPython.display import display, Markdown
        display(Markdown(answer))

    return {"answer": answer, "evidence": evidence, "trace": trace,
            "tools_used": tools_used, "steps": len(trace), "evaluation": evaluation}


print("✓ ReAct agent ready — call run_react_agent(query)")

## Cell 16 — Demo 1: Course-based query

**Goal:** Test that the agent correctly routes to course materials for a lecture concept.

**Expected behaviour:** Agent plans `['course']`, retrieves from lecture PDFs,
produces an answer grounded in actual course content.

In [ ]:
result1 = run_react_agent(
    "Explain the TF-IDF model and how it is used in information retrieval according to our course",
    verbose=True
)
print(f"\nTools used  : {result1['tools_used']}")
print(f"Steps taken : {result1['steps']}")
print(f"Confidence  : {result1['evaluation']['best_similarity']:.3f} similarity")

## Cell 17 — Demo 2: Research-based query

**Goal:** Test that the agent routes to research papers for an academic question.

**Expected behaviour:** Agent plans `['research']`, retrieves from indexed papers,
cites specific paper titles in the answer.

In [ ]:
result2 = run_react_agent(
    "According to the research literature, what are the key limitations of Agentic RAG systems?",
    verbose=True
)
print(f"\nTools used  : {result2['tools_used']}")
print(f"Steps taken : {result2['steps']}")

## Cell 18 — Demo 3: Multi-source query - This is our best demo

### Objective
Evaluate the ability of the Agentic RAG system to combine information from multiple retrieval sources within a single reasoning workflow.

### Expected behavior
The agent:
- identifies the required information sources
- retrieves relevant content from course materials, research papers, and web search
- integrates the retrieved evidence into a unified response

This experiment highlights the advantage of Agentic RAG over traditional single-step RAG systems by demonstrating dynamic tool selection and iterative retrieval.


In [ ]:
result3 = run_react_agent(
    "Explain Agentic RAG: the core concept from our course, "
    "the academic foundations from research papers, "
    "and current real-world applications",
    verbose=True
)
print(f"\nTools used  : {result3['tools_used']}")
print(f"Steps taken : {result3['steps']}")
print(f"Sources     : {len(result3['evidence'])} evidence chunks")

## Cell 19 — Naive RAG vs Agentic RAG comparison

This experiment compares the responses generated by:
1. A traditional Naive RAG pipeline
2. The proposed Agentic RAG framework

### Naive RAG
- Performs a single retrieval step
- Uses only one retrieval source
- Does not include reasoning or iterative refinement

### Agentic RAG
- Supports multi-source retrieval
- Uses iterative reasoning and evidence evaluation
- Dynamically selects additional tools when necessary

The comparison highlights differences in:
- answer completeness
- source diversity
- retrieval quality
- reasoning capability
- overall response reliability

In [ ]:
from IPython.display import display, Markdown

COMPARE_QUESTION = (
    "Compare traditional RAG and Agentic RAG, "
    "and explain why Agentic RAG is better for Web and Text Mining tasks."
)

print("=" * 70)
print("NAIVE RAG — single source, one pass, no reasoning")
print("=" * 70)

# Naive: retrieve top-4 from course only, build prompt, return context as answer
naive_chunks = course_store.search(COMPARE_QUESTION, k=4)
naive_context = "\n\n".join(
    f"[{c['filename']} — chunk {c['chunk_id']}]\n{c['text'][:300]}"
    for c in naive_chunks
)
naive_answer = f"""## Naive RAG Answer

**Retrieved from:** course_materials only (1 source, 1 pass, no reasoning)

{naive_context[:1200]}

**Sources:** {len(naive_chunks)} chunks from course_materials only.
"""
display(Markdown(naive_answer))

print("\n" + "=" * 70)
print("AGENTIC RAG — multi-source, iterative, with reasoning trace")
print("=" * 70)

result_compare = run_react_agent(COMPARE_QUESTION, verbose=True)

# ── Side-by-side metrics ─────────────────────────────────────────────────
print("\n=== COMPARISON SUMMARY ===")
comparison_df = pd.DataFrame([
    {"System": "Naive RAG",    "Sources": 1,
     "Chunks": len(naive_chunks), "Reasoning": "None",
     "Web access": "No",  "Self-correction": "No"},
    {"System": "Agentic RAG",  "Sources": len(set(result_compare["tools_used"])),
     "Chunks": len(result_compare["evidence"]), "Reasoning": "ReAct loop",
     "Web access": "Yes" if "web" in result_compare["tools_used"] else "No",
     "Self-correction": "Yes"},
])
display(comparison_df)

## Cell 20 — Results visualization

This section presents visual analyses of the system behavior across the different experimental scenarios.

### Generated visualizations
1. Tool usage across demos  
   Shows which retrieval sources were selected during each query execution.

2. Retrieved evidence distribution  
   Displays the number of evidence chunks collected for each experiment.

3. Similarity score distribution  
   Illustrates the semantic similarity scores of the retrieved chunks.

4. Execution step comparison  
   Compares the number of reasoning and retrieval steps between Naive RAG and Agentic RAG configurations.

These visualizations help evaluate retrieval efficiency, reasoning behavior, and source utilization within the proposed framework.

In [ ]:
all_results = {
    "Demo 1 — Course query":       result1,
    "Demo 2 — Research query":     result2,
    "Demo 3 — Multi-source":       result3,
    "Compare — Traditional vs Agent": result_compare,
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("ReAct Agentic RAG — Results Analysis", fontsize=14, fontweight="bold")

# ── Chart 1: Tool calls per demo ─────────────────────────────────────────
tool_colors = {"course":"#6366F1","research":"#10B981","web":"#F59E0B"}
tool_names  = ["course","research","web"]
x = np.arange(len(all_results))
width = 0.25
labels = [k.split(" — ")[0] for k in all_results]
for i, tool in enumerate(tool_names):
    vals = [r["tools_used"].count(tool) for r in all_results.values()]
    axes[0,0].bar(x + i*width, vals, width, label=tool,
                  color=tool_colors[tool], alpha=0.85)
axes[0,0].set_xticks(x + width)
axes[0,0].set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
axes[0,0].set_ylabel("Times called")
axes[0,0].set_title("Tool usage per demo")
axes[0,0].legend(fontsize=9)

# ── Chart 2: Evidence chunks collected ────────────────────────────────────
chunk_counts = [len(r["evidence"]) for r in all_results.values()]
axes[0,1].bar(labels, chunk_counts, color="#6366F1", alpha=0.85)
axes[0,1].axhline(4, color="red", linestyle="--", linewidth=1, label="Naive RAG (4 chunks)")
for i, v in enumerate(chunk_counts):
    axes[0,1].text(i, v + 0.3, str(v), ha="center", fontweight="bold")
axes[0,1].set_ylabel("Evidence chunks")
axes[0,1].set_title("Evidence collected per demo")
axes[0,1].legend(fontsize=9)
axes[0,1].set_xticklabels(labels, rotation=15, ha="right", fontsize=9)

# ── Chart 3: Similarity score distribution ────────────────────────────────
for res_label, res in all_results.items():
    sims = [e["similarity"] for e in res["evidence"] if e.get("similarity")]
    if sims:
        axes[1,0].hist(sims, bins=12, alpha=0.55,
                       label=res_label.split(" — ")[0])
axes[1,0].axvline(MIN_SIMILARITY, color="red", linestyle="--",
                  linewidth=1.2, label=f"Threshold ({MIN_SIMILARITY})")
axes[1,0].set_xlabel("Cosine similarity")
axes[1,0].set_ylabel("Count")
axes[1,0].set_title("Similarity score distribution")
axes[1,0].legend(fontsize=8)

# ── Chart 4: Source type coverage ─────────────────────────────────────────
source_coverage = {}
for name, res in all_results.items():
    short = name.split(" — ")[0]
    for t in ["course_material","research_paper","web"]:
        found = any(e.get("source_type")==t for e in res["evidence"])
        source_coverage.setdefault(t, {})[short] = int(found)
cov_df = pd.DataFrame(source_coverage).T
cov_df.plot(kind="bar", ax=axes[1,1], color=["#6366F1","#10B981","#F59E0B","#EC4899"],
            alpha=0.85, legend=True)
axes[1,1].set_xticklabels(cov_df.index, rotation=15, ha="right", fontsize=9)
axes[1,1].set_ylabel("Source found (1=yes, 0=no)")
axes[1,1].set_title("Source coverage per demo")
axes[1,1].set_yticks([0,1])
axes[1,1].set_yticklabels(["Not used","Used"])
axes[1,1].legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig("results_analysis.png", bbox_inches="tight")
plt.show()
print("✓ Plot saved as results_analysis.png")

## Cell 21 — Custom query testing

This section allows testing the Agentic RAG system with additional user-defined queries.

Different types of questions can be evaluated to observe:
- retrieval behavior
- source selection
- reasoning flow
- response generation quality

The system automatically determines which retrieval tools are required based on the query content.

In [ ]:
# ── Change this question ──────────────────────────────────────────────────
MY_QUERY = "What is the difference between BM25 and dense retrieval for information retrieval?"

result_custom = run_react_agent(MY_QUERY, verbose=True)
print(f"\nTools   : {result_custom['tools_used']}")
print(f"Chunks  : {len(result_custom['evidence'])}")
print(f"Steps   : {result_custom['steps']}")

## Cell 22 — Limitations and future work

### Current limitations

| Limitation | Description |
|---|---|
| Rule-based planning | `detect_needs()` uses keyword matching, not LLM reasoning |
| No conversation memory | Each query starts fresh — no multi-turn context |
| Sequential tool calls | Tools run one at a time, not in parallel |
| FAISS rebuilt each run | Index is not persisted to disk |
| Scanned PDF risk | Image-based PDFs yield no extractable text |
| DuckDuckGo removed | Replaced by Tavily — but requires a free API key |

### Future improvements

1. **Add LangGraph** for a proper stateful agent loop with real LLM reasoning at each step
2. **Hybrid retrieval** — combine BM25 (sparse) with dense embeddings for better precision
3. **RAGAS evaluation** — measure faithfulness, answer relevance, context precision against a labelled set
4. **Persist FAISS index** to disk so it does not rebuild on every run
5. **Streamlit UI** — wrap the agent in a simple web interface for non-technical demos
6. **Parallel tool calls** — call all relevant tools simultaneously to reduce latency
7. **Canvas LMS API** — fetch real course materials dynamically instead of loading PDFs manually

## Cell 23 — Conclusion

### What this notebook demonstrates

```
Naive RAG:    Query → Retrieve (once, one source) → Generate
This system:  Thought → Action → Observe → Evaluate → Refine → Synthesize
```

### Key contributions

| Component | What it demonstrates |
|---|---|
| FAISS + SentenceTransformers | Dense semantic retrieval on local documents |
| Three-tool architecture | Multi-source retrieval with source-type separation |
| Evidence evaluation layer | Self-correction — agent decides if evidence is sufficient |
| ReAct trace | Transparent, explainable agentic decision process |
| Answer synthesis from evidence | Grounded answers — not hallucinated templates |
| Visualisation suite | Big Data Analytics mindset — analyse before and after |

### Summary

The proposed system extends traditional Retrieval-Augmented Generation (RAG) by integrating an evidence evaluation and iterative retrieval mechanism.

Instead of relying on a single retrieval step, the agent dynamically selects between multiple information sources, including:
- course materials
- research papers
- live web search

The system then evaluates the retrieved evidence and decides whether additional retrieval steps are necessary before generating the final response.

This approach improves:
- retrieval flexibility
- source diversity
- reasoning capability
- response reliability

A key challenge addressed by the framework is determining when the retrieved evidence is sufficient, which remains an important open research problem in Agentic RAG systems.